### Imports

In [1]:
from pathlib import Path
from typing import Any

import numpy as np

from equity_pricing.calibration import calibrate_smile, calibrate_surface
from equity_pricing.heston import model_surface, price_european
from equity_pricing.plots import plot_residual_heatmap, plot_smile_fit, plot_surface_fit
from equity_pricing.simulation import price_vanilla_mc
from equity_pricing.types import (
    CalibrationSettings,
    FlatMarketInputs,
    HestonParams,
    OptionSide,
    VanillaOption,
)

### Inputs

In [2]:
market = FlatMarketInputs(spot=100.0, risk_free_rate=0.02, dividend_yield=0.01)
true_params = HestonParams(kappa=2.0, theta=0.04, sigma=0.2, rho=-0.3, v0=0.04)
initial_params = HestonParams(kappa=1.4, theta=0.05, sigma=0.35, rho=-0.15, v0=0.05)

maturities = np.array([0.5, 1.0], dtype=float)
strikes_by_expiry = (
    np.array([90.0, 100.0, 110.0], dtype=float),
    np.array([85.0, 100.0, 115.0], dtype=float),
)

### Run

#### Calibrate Equity Vol Surface

In [3]:
# Mock Synthetic surface
synthetic_surface = model_surface(
    strikes_by_expiry,
    maturities,
    market,
    true_params,
    upper_limit=120.0,
    abs_tol=1.0e-7,
    rel_tol=1.0e-7,
)

In [ ]:
calibration_settings = CalibrationSettings(
    upper_limit=120.0,
    abs_tol=1.0e-7,
    rel_tol=1.0e-7,
    integration_limit=150,
    n_restarts=2,
)
surface_result = calibrate_surface(
    synthetic_surface,
    market,
    initial_params,
    calibration_settings,
)
smile_result = calibrate_smile(
    synthetic_surface.smiles[0],
    market,
    initial_params,
    calibration_settings,
)
